In [2]:
import os
import numpy as np
import pandas as pd

import rocks
rocks.set_log_level("error")
import time as t

import requests
import wget


In [3]:
# DAMIT
path_damit = os.path.join('..','..','data','damit')
url = 'https://astro.troja.mff.cuni.cz/projects/damit/generated_files/open/AsteroidModel/'
url_lc = 'https://astro.troja.mff.cuni.cz/projects/damit/LightCurves/lcRef/'

# Read data

In [4]:
for f in ['asteroids', 'asteroid_models', 'references', 'asteroid_models_references']:
    file = os.path.join( path_damit, f'damit-{f}.csv')
    print(file)
    if not os.path.exists(file):
        fname = wget.download(f'https://astro.troja.mff.cuni.cz/projects/damit/exports/table/{f}')
        os.rename(fname, file)

../../data/damit/damit-asteroids.csv
../../data/damit/damit-asteroid_models.csv
../../data/damit/damit-references.csv
../../data/damit/damit-asteroid_models_references.csv


In [5]:
# Read DAMIT Tables
ssos = pd.read_csv(os.path.join( path_damit, 'damit-asteroids.csv'))
models = pd.read_csv(os.path.join( path_damit, 'damit-asteroid_models.csv'))
refs = pd.read_csv(os.path.join( path_damit, 'damit-asteroid_models_references.csv'))
bibs = pd.read_csv(os.path.join( path_damit, 'damit-references.csv'))

In [6]:
# Select columns for ASTEROIDS
ssos = ssos[ ['id', 'number', 'name', 'designation'] ]
ssos.columns = ['asteroid_id', 'number', 'name', 'designation']

In [7]:
# Select columns for MODELS
models = models[ [ 'id', 'asteroid_id', 'lambda', 'beta', 'period' ] ]
models.columns = [ 'model_id', 'asteroid_id', 'lambda', 'beta', 'period' ]

In [8]:
# Select columns for REFERENCES
refs = refs[ ['asteroid_model_id','reference_id'] ]
refs.columns = ['model_id','reference_id']

In [9]:
# Select columns for BIBLIOGRAPHY
bibs = bibs[ ['id', 'bibcode'] ]
bibs.columns = ['reference_id', 'bibcode']

In [10]:
# Merge everything
data = models.merge( ssos, on='asteroid_id')
data = data.merge( refs, on='model_id')
data = data.merge( bibs, on='reference_id')
data

,model_id,asteroid_id,lambda,beta,period,number,name,designation,reference_id,bibcode
0,101,101,35.0,-12.0,7.813230,2.0,Pallas,NaN,106,2003icar..164..346t
1,101,101,35.0,-12.0,7.813230,2.0,Pallas,NaN,139,2011icar..214..652d
2,102,101,32.0,-11.0,7.813220,2.0,Pallas,NaN,169,2017a&a...601a.114h
3,102,101,32.0,-11.0,7.813220,2.0,Pallas,NaN,132,2010icar..205..460c
4,103,102,103.0,27.0,7.209531,3.0,Juno,NaN,103,2002icar..159..369k
...,...,...,...,...,...,...,...,...,...,...
16314,16301,2202,34.0,39.0,16.856900,806.0,Gyldenia,NaN,667,NaN
16315,16302,10862,318.0,60.0,4.463668,138852.0,NaN,NaN,668,2024a&a...682a..93d
16316,16303,10863,260.0,-60.0,7.664354,85989.0,NaN,NaN,668,2024a&a...682a..93d
16317,16304,10864,230.0,36.0,35.984000,357.0,Ninina,NaN,674,2024mpbu...51..100f


In [11]:
data[ data.number==135 ]

,model_id,asteroid_id,lambda,beta,period,number,name,designation,reference_id,bibcode
119,162,142,272.0,52.0,8.4006,135.0,Hertha,NaN,106,2003icar..164..346t
120,162,142,272.0,52.0,8.4006,135.0,Hertha,NaN,127,2009mpbu...36...98t
1846,1799,142,276.0,53.0,8.4006,135.0,Hertha,NaN,169,2017a&a...601a.114h


# LC info and abc

In [12]:
abc = pd.read_csv(os.path.join( path_damit, 'abc.csv'))
lcref = pd.read_csv(os.path.join( path_damit, 'lc_summary.csv'))

In [37]:
lcref

,asteroid_id,N,N_SP
0,10000,1,1
1,10001,1,1
2,10002,1,1
3,10003,1,1
4,10004,1,1
...,...,...,...
10740,9996,1,1
10741,9997,1,1
10742,9998,1,1
10743,9999,1,1


In [46]:
# select asteroids with dense light curves
cond = lcref.N > lcref.N_SP

N_min_LC = 10
cond2 = lcref.N > (lcref.N_SP+N_min_LC)

print(f'Number of asteroids : {len(lcref)}')
print(f'Number with SP only : {len(lcref[~cond])}')
print(f'Number with dense LC: {len(lcref[cond])}')
print(f' with at least {N_min_LC:2d} LC: {len(lcref[cond2])}')



Number of asteroids : 10745
Number with SP only : 9861
Number with dense LC: 884
 with at least 10 LC: 362


In [47]:
targets = lcref[cond2].asteroid_id.to_list()

In [48]:
data[ data.asteroid_id.isin(targets)][20:40]

,model_id,asteroid_id,lambda,beta,period,number,name,designation,reference_id,bibcode
20,111,107,180.0,22.0,5.079176,9.0,Metis,NaN,127,2009mpbu...36...98t
21,111,107,180.0,22.0,5.079176,9.0,Metis,NaN,139,2011icar..214..652d
22,112,108,3.0,-67.0,6.082753,15.0,Eunomia,NaN,103,2002icar..159..369k
23,112,108,3.0,-67.0,6.082753,15.0,Eunomia,NaN,109,2005icar..175..452n
24,112,108,3.0,-67.0,6.082753,15.0,Eunomia,NaN,149,2013icar..226.1045h
25,113,109,32.0,-7.0,4.195948,16.0,Psyche,NaN,103,2002icar..159..369k
26,113,109,32.0,-7.0,4.195948,16.0,Psyche,NaN,139,2011icar..214..652d
27,113,109,32.0,-7.0,4.195948,16.0,Psyche,NaN,149,2013icar..226.1045h
28,116,110,236.0,19.0,12.266030,17.0,Thetis,NaN,123,2009a&a...493..291a
29,116,110,236.0,19.0,12.266030,17.0,Thetis,NaN,139,2011icar..214..652d
